# Task 3: Heart Disease Prediction

**Objective:** Build a binary classifier to predict heart disease risk from patient health data.

**Dataset:** UCI Heart Disease (Cleveland)

**Models:** Logistic Regression, Decision Tree Classifier

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid')

## 1. Load and Clean Dataset

In [ ]:
columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
df = pd.read_csv(url, names=columns, na_values='?')

print('Shape:', df.shape)
print('Missing values:\n', df.isnull().sum())
df.head()

In [ ]:
# Handle missing values with median imputation
df = df.fillna(df.median(numeric_only=True))

# Binary target: 0 = no disease, 1+ = disease present
df['target'] = (df['target'] > 0).astype(int)

print('Target distribution:\n', df['target'].value_counts())
df.describe()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(data=df, x='age', hue='target', kde=True, ax=axes[0, 0], palette='Set1')
sns.histplot(data=df, x='chol', hue='target', kde=True, ax=axes[0, 1], palette='Set1')
sns.boxplot(data=df, x='target', y='thalach', palette='Set1', ax=axes[1, 0])
sns.boxplot(data=df, x='target', y='oldpeak', palette='Set1', ax=axes[1, 1])
plt.suptitle('EDA: Age, Cholesterol, Max Heart Rate, ST Depression', y=1.02)
plt.tight_layout()
plt.show()

## 3. Train Classification Models

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42)
}

results = {}
for name, model in models.items():
    X_tr = X_train_scaled if name == 'Logistic Regression' else X_train
    X_te = X_test_scaled if name == 'Logistic Regression' else X_test
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {'model': model, 'y_pred': y_pred, 'y_prob': y_prob, 'acc': acc, 'auc': auc}
    print(f'{name} — Accuracy: {acc:.4f}, ROC-AUC: {auc:.4f}')
    print(classification_report(y_test, y_pred))
    print()

## 4. Evaluation — Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, res) in zip(axes, results.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, res['y_pred'])).plot(ax=ax, cmap='Blues')
    ax.set_title(f'{name} — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Heart Disease Prediction')
plt.legend()
plt.show()

## 5. Feature Importance

In [ ]:
lr_model = results['Logistic Regression']['model']
dt_model = results['Decision Tree']['model']

importance_df = pd.DataFrame({
    'feature': X.columns,
    'lr_coef': np.abs(lr_model.coef_[0]),
    'dt_importance': dt_model.feature_importances_
}).sort_values('dt_importance', ascending=False)

print('Feature Importance (sorted by Decision Tree):')
importance_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=importance_df, y='feature', x='dt_importance', palette='viridis', ax=ax)
ax.set_title('Decision Tree Feature Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. Results and Insights

- **Key predictors:** chest pain type (`cp`), max heart rate (`thalach`), exercise angina (`exang`), ST depression (`oldpeak`).
- Both models achieve reasonable accuracy; compare ROC-AUC for discrimination ability.
- Higher `oldpeak` and `exang` correlate with increased disease risk.
- **Disclaimer:** Educational model only — not for clinical diagnosis.